# 2.1.0 Train a SciBERT classifier for corpus labeling

This notebook fine-tunes a SciBERT semantic classifier model used for corpus labeling from the GPT-labeled annotation-support corpus seed set.

---

### Configuration and training data

Load the GPT-labeled shadow annotation set, define the training artifact paths, and prepare the train/validation split for SciBERT fine-tuning.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def _find_workspace_root() -> Path:
    env_root = os.environ.get("WORKSPACE_ROOT", "").strip()
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    for start in [Path.cwd(), *Path.cwd().parents]:
        if (start / "config" / "workspace.json").exists():
            return start
    raise FileNotFoundError(
        "Could not locate workspace root from WORKSPACE_ROOT or the current working directory."
    )


WORKSPACE_DIR = _find_workspace_root()
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

CODE_DIR = WORKSPACE_DIR / "code"
NOTEBOOKS_DIR = CODE_DIR / "notebooks"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
SHARED_ASSETS_DIR = paths["shared_assets"]
NOTEBOOKS_DIR = CODE_DIR / "notebooks"

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
from transformers import AutoModel, AutoTokenizer, DataCollatorWithPadding, get_linear_schedule_with_warmup

from notebook_theme import activate_theme, colors

theme_name = "paper"
theme = activate_theme(theme_name)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SHADOW_SET_ID = "shadow_balanced_1800"
PROMPT_VERSION = 5
SEED_RUN_ID = f"{SHADOW_SET_ID}__gpt54_prompt_v{PROMPT_VERSION}"
SCIBERT_RUN_ID = "scibert_classifier_3labs"
RUN_MODE = "train"  # "replay" | "train"
SEED_LABELS_RUN_TAG = "full_run"  # "run_1" | "run_2" | "full_run"

SEED_LABELS_N = 1800 if SEED_LABELS_RUN_TAG == "full_run" else 200
SEED_LABELS_PATH = (
    OUTPUTS_DIR
    / "analytical-results"
    / "models"
    / "gpt"
    / "runs"
    / SEED_RUN_ID
    / f"annotation_labeled_{SEED_LABELS_N}_{SEED_LABELS_RUN_TAG}.jsonl"
)
SCIBERT_DIR = OUTPUTS_DIR / "analytical-results" / "models" / "scibert"
SCIBERT_BASE = LOCAL_DIR / "models" / "scibert_scivocab_uncased"
BEST_MODEL_PATH = SCIBERT_DIR / f"{SCIBERT_RUN_ID}_best.pt"
FINAL_MODEL_PATH = SCIBERT_DIR / f"{SCIBERT_RUN_ID}.pt"
TRAINING_LOG_PATH = SCIBERT_DIR / "training_log.json"
THRESHOLDS_PATH = SCIBERT_DIR / f"{SCIBERT_RUN_ID}_thresholds.json"
LABEL_ORDER_PATH = SCIBERT_DIR / f"{SCIBERT_RUN_ID}_label_order.json"
TRAINING_CURVES_PATH = SCIBERT_DIR / "training_curves.png"
SCIBERT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
LABELS = ["A", "B", "C"]
MAX_LEN = 128
TRAIN_BATCH_SIZE = 8
VAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MAX_EPOCHS = 10
PATIENCE = 2
THRESHOLD_GRID = np.arange(0.3, 0.71, 0.05)

if not SEED_LABELS_PATH.exists():
    raise FileNotFoundError(
        f"Missing seed labels at {SEED_LABELS_PATH}. "
        "Run 2.0.0-seed-classification-gpt.ipynb for the selected run tag first."
    )

with open(SEED_LABELS_PATH, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

records = [row for row in records if row.get("labels")]
texts = [row.get("sentence_marked") or row.get("sentence") or row.get("sent", "") for row in records]
labels_raw = [row["labels"] for row in records]

mlb = MultiLabelBinarizer(classes=LABELS)
texts_train, texts_val, Y_train, Y_val = train_test_split(
    texts,
    mlb.fit_transform(labels_raw).astype(np.float32),
    test_size=0.2,
    random_state=42,
)

print("SEED_LABELS_PATH:", SEED_LABELS_PATH, "exists:", SEED_LABELS_PATH.exists())
print("SCIBERT_DIR:", SCIBERT_DIR)
print("Using device:", DEVICE)
print(f"Train: {len(texts_train)}, Val: {len(texts_val)}")


### Helper classes and evaluation functions

---

In [ ]:
class MentionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = texts
        self.labels = torch.tensor(labels, dtype=torch.float)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": self.labels[idx],
        }


class SciBERTClassifier(nn.Module):
    def __init__(self, base_model, n_labels=3, dropout=0.2):
        super().__init__()
        self.bert = base_model
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls))


def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            logits = model(input_ids, attention_mask)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(batch["labels"].numpy())

    Y_prob = np.vstack(all_probs)
    Y_true = np.vstack(all_labels).astype(int)
    Y_pred = (Y_prob >= threshold).astype(int)

    print(classification_report(Y_true, Y_pred, target_names=LABELS, zero_division=0))
    acc = accuracy_score(Y_true, Y_pred)
    print(f"  Exact-match accuracy: {acc:.4f}")

    try:
        roc = roc_auc_score(Y_true, Y_prob, average=None)
        roc_macro = roc_auc_score(Y_true, Y_prob, average="macro")
        for label, score in zip(LABELS, roc):
            print(f"  ROC-AUC {label}: {score:.4f}")
        print(f"  ROC-AUC macro: {roc_macro:.4f}")
    except ValueError as e:
        print(f"  ROC-AUC skipped: {e}")
        roc_macro = None

    return Y_pred, Y_true, Y_prob, acc, roc_macro


def threshold_sweep(Y_true, Y_prob, thresholds=THRESHOLD_GRID):
    print("\n  Threshold sweep (macro F1):")
    best_thresh, best_f1 = 0.5, 0.0
    best_by_label = {}

    for thresh in thresholds:
        Y_pred = (Y_prob >= thresh).astype(int)
        f1 = f1_score(Y_true, Y_pred, average="macro", zero_division=0)
        marker = " <--" if f1 > best_f1 else ""
        print(f"    thresh={thresh:.2f}  macro_f1={f1:.4f}{marker}")
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh

    print("\n  Per-label optimal threshold:")
    for i, label in enumerate(LABELS):
        best_lt, best_lf = 0.5, 0.0
        for thresh in thresholds:
            pred = (Y_prob[:, i] >= thresh).astype(int)
            f1 = f1_score(Y_true[:, i], pred, zero_division=0)
            if f1 > best_lf:
                best_lf, best_lt = f1, thresh
        best_by_label[label] = {"threshold": float(best_lt), "f1": float(best_lf)}
        print(f"    {label}: thresh={best_lt:.2f}  f1={best_lf:.4f}")

    return {"macro": {"threshold": float(best_thresh), "f1": float(best_f1)}, "per_label": best_by_label}


## Train classifier

The training epochs include an autosave for the best performing model that doesn't overfit.

---

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(SCIBERT_BASE, use_fast=True)
base_model = AutoModel.from_pretrained(SCIBERT_BASE).to(DEVICE)
print(f"Loaded SciBERT on {DEVICE}")

train_dataset = MentionDataset(texts_train, Y_train, tokenizer)
val_dataset = MentionDataset(texts_val, Y_val, tokenizer)
collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_dataset, batch_size=VAL_BATCH_SIZE, shuffle=False, collate_fn=collator)

model = SciBERTClassifier(base_model, n_labels=len(LABELS), dropout=0.2).to(DEVICE)

pos_counts = Y_train.sum(axis=0)
neg_counts = len(Y_train) - pos_counts
pos_weights = torch.tensor(
    np.clip(neg_counts / np.maximum(pos_counts, 1), 1.0, None),
    dtype=torch.float,
    device=DEVICE,
)

print("Positive counts:", pos_counts)
print("Negative counts:", neg_counts)
print("pos_weights:", pos_weights)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * MAX_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps,
)

training_log = []
best_macro_f1 = 0.0
best_epoch = 0
no_improve_epochs = 0

if RUN_MODE == "replay":
    required_artifacts = [BEST_MODEL_PATH, TRAINING_LOG_PATH, THRESHOLDS_PATH, LABEL_ORDER_PATH]
    missing_artifacts = [path for path in required_artifacts if not path.exists()]
    if missing_artifacts:
        missing_text = "\n  - ".join(str(path) for path in missing_artifacts)
        raise FileNotFoundError(
            "RUN_MODE='replay' requires existing training artifacts. Missing:\n  - "
            f"{missing_text}"
        )

    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=True))
    with open(TRAINING_LOG_PATH, "r", encoding="utf-8") as f:
        training_log = json.load(f)
    with open(THRESHOLDS_PATH, "r", encoding="utf-8") as f:
        threshold_info = json.load(f)
    with open(LABEL_ORDER_PATH, "r", encoding="utf-8") as f:
        saved_label_order = json.load(f)

    if saved_label_order != list(mlb.classes_):
        raise ValueError(
            "Saved label order does not match current label space. "
            f"Saved: {saved_label_order} | current: {list(mlb.classes_)}"
        )

    best_epoch = int(threshold_info["best_epoch"])
    best_macro_f1 = float(threshold_info["best_macro_f1"])
    print("Loaded cached SciBERT training artifacts.")
    print("Best model:", BEST_MODEL_PATH)
    print("Training log:", TRAINING_LOG_PATH)
    print("Thresholds:", THRESHOLDS_PATH)
    print("Label order:", LABEL_ORDER_PATH)

elif RUN_MODE == "train":
    for epoch in range(MAX_EPOCHS):
        model.train()
        total_loss, batch_losses = 0.0, []

        loop = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{MAX_EPOCHS}", leave=True)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            batch_losses.append(loss.item())
            loop.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{MAX_EPOCHS} — avg loss: {avg_loss:.4f}")

        Y_pred_ep, Y_true_ep, Y_prob_ep, acc_ep, roc_ep = evaluate(model, val_loader)
        macro_f1 = f1_score(Y_true_ep, Y_pred_ep, average="macro", zero_division=0)
        report = classification_report(
            Y_true_ep,
            Y_pred_ep,
            target_names=LABELS,
            zero_division=0,
            output_dict=True,
        )

        epoch_record = {
            "epoch": epoch + 1,
            "avg_loss": float(avg_loss),
            "batch_losses": [float(loss) for loss in batch_losses],
            "macro_f1": float(macro_f1),
            "accuracy": float(acc_ep),
            "roc_auc_macro": float(roc_ep) if roc_ep is not None else None,
            "val_report": report,
            "best": False,
            "patience_counter": int(no_improve_epochs),
        }
        training_log.append(epoch_record)

        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            best_epoch = epoch + 1
            no_improve_epochs = 0
            epoch_record["best"] = True
            epoch_record["patience_counter"] = 0
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"  ✓ New best macro F1: {macro_f1:.4f} — checkpoint saved to {BEST_MODEL_PATH}")
        else:
            no_improve_epochs += 1
            epoch_record["patience_counter"] = int(no_improve_epochs)
            print(f"  No improvement. patience={no_improve_epochs}/{PATIENCE}")
            if no_improve_epochs >= PATIENCE:
                print(f"  Early stopping triggered at epoch {epoch + 1}. Best epoch: {best_epoch}")
                break

    torch.save(model.state_dict(), FINAL_MODEL_PATH)
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=True))
    _, Y_true_best, Y_prob_best, _, _ = evaluate(model, val_loader)
    threshold_info = threshold_sweep(Y_true_best, Y_prob_best)
    threshold_info["best_epoch"] = int(best_epoch)
    threshold_info["best_macro_f1"] = float(best_macro_f1)

    with open(TRAINING_LOG_PATH, "w", encoding="utf-8") as f:
        json.dump(training_log, f, indent=2)
    with open(LABEL_ORDER_PATH, "w", encoding="utf-8") as f:
        json.dump(list(mlb.classes_), f, indent=2)
    with open(THRESHOLDS_PATH, "w", encoding="utf-8") as f:
        json.dump(threshold_info, f, indent=2)

    print("Model and training artifacts saved.")
    print("Best epoch:", best_epoch)
    print("Best model:", BEST_MODEL_PATH)
    print("Final model:", FINAL_MODEL_PATH)
    print("Training log:", TRAINING_LOG_PATH)
    print("Thresholds:", THRESHOLDS_PATH)
    print("Label order:", LABEL_ORDER_PATH)

else:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE!r}. Use 'replay' or 'train'.")


## Review training

Inspect training curves and validation-threshold behavior of the model.

---

In [ ]:
if "training_log" not in globals() or "threshold_info" not in globals():
    with open(TRAINING_LOG_PATH, "r", encoding="utf-8") as f:
        training_log = json.load(f)
    with open(THRESHOLDS_PATH, "r", encoding="utf-8") as f:
        threshold_info = json.load(f)

epochs = [row["epoch"] for row in training_log]
avg_losses = [row["avg_loss"] for row in training_log]
macro_f1s = [row["macro_f1"] for row in training_log]
a_f1s = [row["val_report"]["A"]["f1-score"] for row in training_log]
b_f1s = [row["val_report"]["B"]["f1-score"] for row in training_log]
c_f1s = [row["val_report"]["C"]["f1-score"] for row in training_log]
best_flags = [bool(row.get("best")) for row in training_log]
best_epoch = int(threshold_info["best_epoch"])
best_macro_f1 = float(threshold_info["best_macro_f1"])
stopped_epoch = epochs[-1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
labels = {"A": "A | Referential", "B": "B | Role", "C": "C | Model-parameter"}

ax1.plot(epochs, avg_losses, marker="o", color="steelblue")
ax1.axvline(best_epoch, color="black", linestyle="--", linewidth=1, alpha=0.8, label=f"Best epoch = {best_epoch}")
if stopped_epoch != best_epoch:
    ax1.axvline(stopped_epoch, color="firebrick", linestyle=":", linewidth=1.2, alpha=0.9, label=f"Stopped at {stopped_epoch}")
ax1.scatter(
    [epochs[i] for i, is_best in enumerate(best_flags) if is_best],
    [avg_losses[i] for i, is_best in enumerate(best_flags) if is_best],
    color="black",
    s=45,
    zorder=3,
)
ax1.set_title("Training loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Avg BCE loss")
ax1.grid(alpha=0.3)
ax1.legend()

ax2.plot(epochs, macro_f1s, marker="o", label="Macro F1", color="#fcf8e6", lw=1.5)
ax2.plot(epochs, a_f1s, marker="s", label=labels["A"], color=colors["A"])
ax2.plot(epochs, b_f1s, marker="s", label=labels["B"], color=colors["B"])
ax2.plot(epochs, c_f1s, marker="s", label=labels["C"], color=colors["C"])

ax2.axvline(best_epoch, color=theme["fg"], linestyle="--", linewidth=1, alpha=0.8)
if stopped_epoch != best_epoch:
    ax2.axvline(stopped_epoch, color="firebrick", linestyle=":", linewidth=1.2, alpha=0.9)

ax2.scatter([best_epoch], [best_macro_f1], color=theme["fg"], s=55, zorder=4)
ax2.annotate(
    f"best epoch {best_epoch}\nmacro F1 = {best_macro_f1:.3f}",
    xy=(best_epoch, best_macro_f1),
    xytext=(best_epoch - 2.0, best_macro_f1 + 0.1),
    fontsize=9,
    arrowprops={"arrowstyle": "->", "lw": 0.8, "color": theme["fg"]},
)

if stopped_epoch != best_epoch:
    ax2.annotate(
        f"early stop at {stopped_epoch}",
        xy=(stopped_epoch, macro_f1s[-1]),
        xytext=(stopped_epoch - 1.5, macro_f1s[-1] + 0.1),
        fontsize=9,
        color=theme["fg"],
        arrowprops={"arrowstyle": "->", "lw": 0.8, "color": theme["fg"]},
    )

ax2.set_title("Validation F1 by category")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("F1 score")
ax2.set_ylim(0.5, 1.0)
ax2.grid(alpha=0.3)
ax2.legend()

plt.tight_layout()
pdf_path = TRAINING_CURVES_PATH.with_suffix(".pdf")
plt.savefig(TRAINING_CURVES_PATH, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")
print(f"Saved figure -> {TRAINING_CURVES_PATH}")
print(f"Saved figure -> {pdf_path}")
plt.show()
